# Phase 3 · Synthesizer Benchmark (CTAB-GAN+ Protocol)

**Goal**: Compare multiple generative models to see which one learns the real data distribution best. 
We benchmark across 5 architectures following the methodology in the CTAB-GAN+ paper:

1. CTGAN
2. TVAE
3. CopulaGAN
4. Gaussian Copula
5. TabDDPM (Diffusion)

**Evaluation Protocol**:
- 80/20 train/test split on real data.
- Fit synthesizers on the training set.
- Generate a synthetic set equal in size to the training set.
- Evaluate Statistical Similarity (Average JSD, Average Wasserstein Distance, Correlation Difference).
- Evaluate ML Utility (Accuracy, F1, AUC across 5 classification models evaluated on the real test set).
- Report aggregated metrics across N random seeds.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 1 · Imports & Environment Setup
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, logging, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('phase3_benchmark')

# ── Resolve project root ──────────────────────────────────────────────────────
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while not (PROJECT_ROOT / 'AGENTS.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
OUTPUT_DIR  = PROJECT_ROOT / 'docs' / 'generator_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('\n✓ Imports complete')

## 1. Load Real Dataset for Benchmarking

In [ ]:
# We benchmark on the ML-ready feature set derived from real data.
# For this notebook, we'll use a subset if the data is large to speed up evaluation.
# Note: This expects a cleaned real dataset with features and a target.
# Since we are building the pipeline, we'll load the interim canonical transactions
# and construct a small feature set for benchmarking.

from src.generation.core.feature_engineer import engineer_features
from src.generation.core.enricher import enrich_transactions

# Load canoncial data
tx_real = pd.read_parquet(INTERIM_DIR / 'transactions.parquet')
acc_real = pd.read_parquet(INTERIM_DIR / 'accounts.parquet')

# Use a random sample for the benchmark to keep runtime manageable
SAMPLE_SIZE = 10_000
if len(tx_real) > SAMPLE_SIZE:
    tx_sample = tx_real.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    tx_sample = tx_real.copy()

print(f"Using {len(tx_sample):,} rows for benchmark.")

# Enrich and Feature Engineer (determinstic mapping to ML format)
enriched = enrich_transactions(tx_sample, acc_real)
ml_features_real = engineer_features(enriched)

# Drop columns that are IDs, pure dates, or internal metadata, keeping only ML features
cols_to_drop = ['Sender_account', 'Receiver_account', 'Date', 'Time', 
                'transaction_id', 'sender_account_id', 'receiver_account_id',
                '_validation_passed', '_violation_codes', 'original_currency']

ml_data = ml_features_real.drop(columns=[c for c in cols_to_drop if c in ml_features_real.columns])

TARGET_COL = 'is_suspicious_tx'
print(f"\nBenchmark dataset shape: {ml_data.shape}")
print(f"Columns: {list(ml_data.columns)}")
print(f"Target '{TARGET_COL}' distribution:\n{ml_data[TARGET_COL].value_counts(normalize=True)}")

## 2. Initialize Synthesizers

In [ ]:
from src.generation.synthesizers.ctgan_generator import CTGANGenerator
from src.generation.synthesizers.tvae_generator import TVAEGenerator
from src.generation.synthesizers.copulagan_generator import CopulaGANGenerator
from src.generation.synthesizers.gaussian_copula_generator import GaussianCopulaGenerator
from src.generation.synthesizers.tabddpm_generator import TabDDPMGenerator

# We instantiate each generator. They all implement the BaseGenerator interface.
# Note: TabDDPM requires synthcity to be installed. We gracefully handle if it's missing.

synthesizers = {
    "GaussianCopula": GaussianCopulaGenerator(config={}),
    "CTGAN": CTGANGenerator(config={'epochs': 50}),  # Reduced epochs for speed in benchmark
    "TVAE": TVAEGenerator(config={'epochs': 50}),
    "CopulaGAN": CopulaGANGenerator(config={'epochs': 50}),
}

try:
    synthesizers["TabDDPM"] = TabDDPMGenerator(config={})
except Exception as e:
    print(f"TabDDPM skipped: {e}")
    
print("Synthesizers ready for evaluation:")
for name in synthesizers.keys():
    print(f" - {name}")

## 3. Run Benchmark

In [ ]:
from src.evaluation.benchmark_runner import BenchmarkRunner

# Configure the benchmark
# In a real run, you'd use multiple seeds (e.g., [42, 0, 1]). 
# We use [42] here for a faster notebook execution.
runner = BenchmarkRunner(
    real_data=ml_data,
    target_col=TARGET_COL,
    seeds=[42],
    test_size=0.2,
    output_dir=OUTPUT_DIR
)

# Run the full pipeline
results = runner.run(synthesizers)

print("\n✓ Benchmark complete")

## 4. Final Comparison Report

In [ ]:
from src.evaluation.report import build_final_table

# Build and save the final table (CSV + Markdown)
final_df = build_final_table(results, output_dir=OUTPUT_DIR)

# Display nicely in the notebook
final_df

## 5. Extended Benchmark — CTAB-GAN+ & TabSyn

Run this section independently; it does **not** require re-running cells 1–4.
`ml_data` and `OUTPUT_DIR` must already be defined (run cells 1–2 once).

Results are saved to `docs/generator_benchmark/<Model>/metrics.json` and
the consolidated tables are rebuilt automatically.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 5 · CTAB-GAN+ & TabSyn benchmark  (paste-and-run — no re-computation needed)
# ──────────────────────────────────────────────────────────────────────────────
import json
from src.generation.synthesizers.ctabganplus_generator import CTABGANPlusGenerator
from src.evaluation.benchmark_runner import BenchmarkRunner, SynthesizerResult
from src.evaluation.report import build_final_table

# ── TabSyn inline wrapper (synthyverse) ───────────────────────────────────────
class _TabSynWrapper:
    """Adapts synthyverse.TabSynGenerator to the BaseGenerator interface."""
    def __init__(self, target_column, vae_epochs=200, diff_epochs=500):
        from synthyverse.generators.tabsyn_generator import TabSynGenerator as _Syn
        self._cls = _Syn
        self.target_column = target_column
        self.vae_epochs = vae_epochs
        self.diff_epochs = diff_epochs
        self.model = None
        self.is_fitted = False
        self._columns = None

    def fit(self, data):
        self._columns = data.columns.tolist()
        discrete = [
            c for c in data.columns
            if data[c].dtype == 'object' or data[c].dtype.name == 'category' or data[c].nunique() <= 20
        ]
        target = self.target_column if self.target_column in data.columns else (discrete[0] if discrete else data.columns[0])
        self.model = self._cls(target_column=target, vae_num_epochs=self.vae_epochs, epochs=self.diff_epochs)
        self.model.fit(data, discrete)
        self.is_fitted = True

    def generate(self, num_rows):
        syn = self.model.generate(num_rows)
        if self._columns:
            for c in self._columns:
                if c not in syn.columns: syn[c] = 0
            syn = syn[[c for c in self._columns if c in syn.columns]]
        return syn

    def save(self, path): raise NotImplementedError
    @classmethod
    def load(cls, path): raise NotImplementedError


# ── Initialise synthesizers ───────────────────────────────────────────────────
extra_synthesizers = {}

try:
    extra_synthesizers['CTABGANPlus'] = CTABGANPlusGenerator(epochs=150)
    print('✓ CTAB-GAN+ ready')
except Exception as e:
    print(f'CTAB-GAN+ skipped: {e}')

try:
    extra_synthesizers['TabSyn'] = _TabSynWrapper(target_column=TARGET_COL, vae_epochs=200, diff_epochs=500)
    print('✓ TabSyn ready')
except Exception as e:
    print(f'TabSyn skipped: {e}')

# ── Run benchmark ─────────────────────────────────────────────────────────────
extra_runner = BenchmarkRunner(
    real_data=ml_data,
    target_col=TARGET_COL,
    seeds=[42],
    test_size=0.2,
    output_dir=OUTPUT_DIR,   # saves <Model>/metrics.json automatically
)
extra_results = extra_runner.run(extra_synthesizers)
print('\n✓ Extra benchmark complete')

# ── Rebuild final tables from ALL metrics.json files ─────────────────────────
# This merges new results with existing ones (CTGAN, TVAE, CopulaGAN, GaussianCopula, TabDDPM)
all_results = {}
for synth_dir in sorted(OUTPUT_DIR.iterdir()):
    if not synth_dir.is_dir(): continue
    mf = synth_dir / 'metrics.json'
    if not mf.exists(): continue
    with open(mf) as f:
        data = json.load(f)
    name = data['synthesizer']
    r = SynthesizerResult(synthesizer_name=name)
    for s in data['seeds']:
        r.seeds_run.append(s['seed'])
        r.raw.append(s)
    all_results[name] = r

final_df = build_final_table(all_results, output_dir=OUTPUT_DIR)
print(f'\nUpdated tables saved to {OUTPUT_DIR}')
final_df